In [ ]:
# Install openpyxl (required for Excel reading in JupyterLite/Pyodide)
import sys
try:
    import openpyxl
except ImportError:
    if "pyodide" in sys.modules or hasattr(__builtins__, "__BRYTHON__"):
        import micropip
        await micropip.install("openpyxl")
    else:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from pathlib import Path
from openpyxl import load_workbook

# ================================================================
# Auto-detect and load the Excel data sheet
# ================================================================
def find_excel_file():
    """Search common locations for the lab Excel file."""
    search_dirs = [Path("/content"), Path.cwd(), Path("/")]
    candidates = []
    for base in search_dirs:
        if base.exists():
            candidates.extend(base.glob("*流動*.xlsx"))
            candidates.extend(base.glob("*lab*.xlsx"))
            candidates.extend(base.glob("*.xlsx"))
    unique, seen = [], set()
    for p in candidates:
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    if not unique:
        raise FileNotFoundError(
            "No Excel file found. Please upload your .xlsx data sheet "
            "to the JupyterLite file browser, then re-run this cell."
        )
    print(f"✅ Using Excel file: {unique[0]}")
    return str(unique[0])


def read_col(ws, col, row_start, row_end):
    """Return a column range as a NumPy float array."""
    values = [ws[f"{col}{r}"].value for r in range(row_start, row_end + 1)]
    if any(val is None for val in values):
        raise ValueError(
            f"Empty cell found in column {col}, rows {row_start}–{row_end}. "
            "Please check your data sheet."
        )
    return np.array(values, dtype=float)


# ── Load workbook ────────────────────────────────────────────────
excel_path = find_excel_file()
wb = load_workbook(excel_path, data_only=True)
ws = wb["Lab-turbulent-flow"]

# ── Physical properties (read from sheet header) ─────────────────
rho  = float(ws["B8"].value)   # kg/m^3, air density
mu   = float(ws["C8"].value)   # Pa s,   air viscosity
rhow = float(ws["B9"].value)   # kg/m^3, water density
g    = float(ws["E8"].value)   # m/s^2,  gravitational acceleration

# ── Tube geometry ────────────────────────────────────────────────
D = float(ws["A22"].value)     # m, inner diameter
L = float(ws["B22"].value)     # m, tube length

# ── Measured data (rows 26–31) ───────────────────────────────────
Q_meas = read_col(ws, "B", 26, 31)   # m^3/h, measured volumetric flow rate
dh_cm  = read_col(ws, "D", 26, 31)   # cm,    measured manometer height difference

# ── Measurement uncertainties ────────────────────────────────────
sigma_Q  = float(ws["B33"].value)    # m^3/h, absolute uncertainty in Q
sigma_dh = float(ws["D33"].value)    # cm,    absolute uncertainty in Δh

# ── Unit conversions ─────────────────────────────────────────────
Q_m3s = Q_meas / 3600.0                    # m^3/s
v     = 4.0 * Q_m3s / (np.pi * D**2)      # m/s,  cross-section mean velocity
dh    = dh_cm * 1e-2                        # m,    height difference

# ── Relative errors (%) for error analysis ───────────────────────
# sigma_Q = Q_max * 5%  and  sigma_dh = dh_max * 5%  (Excel: B33=B31*0.05, D33=D31*0.05)
# → constant 5% relative error, the same for every data point
error_in_v  = sigma_Q  / Q_meas.max() * 100   # % (= 5% for all points)
error_in_dh = sigma_dh / dh_cm.max()  * 100   # % (= 5% for all points)

print(f"Tube: D = {D*1e3:.2f} mm, L = {L:.3f} m")
print(f"rho = {rho} kg/m³, mu = {mu:.2e} Pa·s, rho_water = {rhow} kg/m³, g = {g} m/s²")
print(f"\nLoaded {len(v)} data points:")
print(f"  v      = {np.round(v,  3)} m/s")
print(f"  dh     = {np.round(dh, 4)} m")
print(f"  err_v  = {np.round(error_in_v,  2)} %")
print(f"  err_dh = {np.round(error_in_dh, 2)} %")

# ================================================================
# Derived quantities
# ================================================================
Re    = v * D * rho / mu
P     = rhow * g * dh
fexpt = P / (2 * rho * v**2 * (L / D))

def calculate_error_bounds(v, dh):
    v_lower  = v  * (1 - error_in_v  / 100)
    v_upper  = v  * (1 + error_in_v  / 100)
    dh_lower = dh * (1 - error_in_dh / 100)
    dh_upper = dh * (1 + error_in_dh / 100)

    Re_lower = v_lower * D * rho / mu
    Re_upper = v_upper * D * rho / mu

    P_lower  = rhow * g * dh_lower
    P_upper  = rhow * g * dh_upper

    f_lower  = P_lower  / (2 * rho * v_upper**2 * (L / D))
    f_upper  = P_upper  / (2 * rho * v_lower**2 * (L / D))

    return Re_lower, Re_upper, f_lower, f_upper

Re_lower, Re_upper, f_lower, f_upper = calculate_error_bounds(v, dh)

f_theory = lambda Re: 0.0791 * Re**(-0.25)

def f_model(Re, C1, C2):
    return C1 * Re**C2

initial_guess = [0.08, -0.25]
params, covariance = curve_fit(f_model, Re, fexpt, p0=initial_guess)
C1, C2 = params
errors = np.sqrt(np.diag(covariance))
C1_error, C2_error = errors

# ── Monte Carlo simulation ───────────────────────────────────────
def monte_carlo_analysis(v, dh, num_simulations=1000):
    np.random.seed(42)
    synthetic_C1 = []
    synthetic_C2 = []

    for _ in range(num_simulations):
        v_simulated  = v  + np.random.normal(0, error_in_v  / 100 * v,  size=len(v))
        dh_simulated = dh + np.random.normal(0, error_in_dh / 100 * dh, size=len(dh))

        Re_simulated = v_simulated * D * rho / mu
        P_simulated  = rhow * g * dh_simulated
        f_simulated  = P_simulated / (2 * rho * v_simulated**2 * (L / D))

        try:
            params_sim, _ = curve_fit(f_model, Re_simulated, f_simulated, p0=initial_guess)
            synthetic_C1.append(params_sim[0])
            synthetic_C2.append(params_sim[1])
        except Exception:
            continue

    return np.array(synthetic_C1), np.array(synthetic_C2)

synthetic_C1, synthetic_C2 = monte_carlo_analysis(v, dh)

C1_lower = np.percentile(synthetic_C1, 2.5)
C1_upper = np.percentile(synthetic_C1, 97.5)
C2_lower = np.percentile(synthetic_C2, 2.5)
C2_upper = np.percentile(synthetic_C2, 97.5)

params_summary = pd.DataFrame({
    "Parameter":    ["C1", "C2"],
    "Value":        [round(C1, 4), round(C2, 3)],
    "95% CI Lower": [round(C1_lower, 3), round(C2_lower, 3)],
    "95% CI Upper": [round(C1_upper, 3), round(C2_upper, 3)],
})

# ── Plots ────────────────────────────────────────────────────────
plt.figure(figsize=(5, 5))
plt.errorbar(Re, fexpt,
             xerr=[Re - Re_lower, Re_upper - Re],
             yerr=[fexpt - f_lower, f_upper - fexpt],
             fmt='o', markersize=8, color='tab:blue', alpha=1, capsize=3)
plt.plot(Re, f_model(Re, C1, C2),  color='tab:blue',   linestyle='-',
         label=r'Best-fit model $f = C_1 \mathrm{Re}^{C_2}$')
plt.plot(Re, f_theory(Re),          color='tab:orange', linestyle='--',
         label=r'$f_{\mathrm{theory}} = 0.0791\,\mathrm{Re}^{-0.25}$')

plt.title('Analysis of friction factor for turbulent air flow in a plastic tube', pad=20)
plt.xlabel('Reynolds number (Re)')
plt.ylabel('Friction factor ($f$)')
plt.ylim(0.0, 0.02)
plt.xlim(0, 10000)
plt.yticks(np.arange(0.0, 0.021, 0.005))
plt.minorticks_on()
plt.grid(True, which='both',  linestyle='--', linewidth=0.5)
plt.grid(True, which='minor', linestyle=':',  linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("friction_factor_plot.png", dpi=300)
plt.show()

plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
plt.hist(synthetic_C1, bins=30, color='tab:olive', alpha=0.7, edgecolor='black')
plt.axvline(C1, color='red', linestyle='--', label=f'Best-fit $C_1 = {C1:.4f}$')
plt.title('Distribution of $C_1$')
plt.xlabel('$C_1$')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(synthetic_C2, bins=30, color='tab:cyan', alpha=0.7, edgecolor='black')
plt.axvline(C2, color='red', linestyle='--', label=f'Best-fit $C_2 = {C2:.3f}$')
plt.title('Distribution of $C_2$')
plt.xlabel('$C_2$')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

params_summary


In [ ]:
# Uncomment to download the plot on Google Colab
# from google.colab import files
# files.download("friction_factor_vs_reynolds.png")
# files.download("distribution_C1.png")
# files.download("distribution_C2.png")
